# ragleap-physical: Phase 3 demo (simulation only)
An AI employee reads a simulator over MCP, asks for a proposal, a named person approves it, the command runs on a real physics engine (MuJoCo), and the outcome is checked on the simulator state.

1. Upload `ragleap-physical.zip` in the Files panel. 2. Runtime > Run all. 3. Copy ALL the output (and save `phase3-demo.png`).

**Limits:** simulation only; one joint; the approval here is done by code standing in for a person; this is not RagLeap's own approval gate or MCP client yet.

In [ ]:
!unzip -q -o ragleap-physical.zip && cd ragleap-physical && pip install -q ".[mujoco]" matplotlib && echo installed

## Start the bridge and the MCP server

In [ ]:
# STEP 0: start the bridge and its MCP server (simulation only)
import dataclasses, json, threading, urllib.request
from ragleap_physical import ApprovalError, Bridge, Capability, Envelope, PhysicalTools, SafetyStop
from ragleap_physical.adapters import Adapter, FakeJointAdapter
from ragleap_physical.mcp_server import make_server

USE_FAKE = globals().get("USE_FAKE", False)   # False = real MuJoCo engine (the default in Colab)

ENVELOPE = {
    "envelope_version": 0, "allowed_commands": ["set_position_target"],
    "targets": {"joint_1": {"pos_min": -1.0, "pos_max": 1.0, "vel_max": 10.0, "force_max": 10.0}},
    "rate": {"max_commands_per_s": 5}, "episode": {"max_sim_time_s": 60, "max_steps": 100000},
    "on_violation": "reject",
}


class RecordingAdapter(Adapter):
    """Wraps an adapter and records (time, position) so we can plot it. Safety is NOT done here."""

    def __init__(self, inner):
        self.inner, self.trace = inner, []
        self._sample()

    def _sample(self):
        raw = self.inner.observe_raw()
        self.trace.append((raw["sim_time_s"], raw["entities"][0]["pos"]))

    simulated = property(lambda self: self.inner.simulated)
    observe_raw = lambda self: self.inner.observe_raw()
    dry_run = lambda self, *a: self.inner.dry_run(*a)
    apply = lambda self, *a: self.inner.apply(*a)
    stop = lambda self: self.inner.stop()
    clear_stop = lambda self: self.inner.clear_stop()

    def advance(self, to_sim_time_s, max_steps):
        ok = True
        while ok:
            now = self.inner.observe_raw()["sim_time_s"]
            if now >= to_sim_time_s - 1e-12:
                break
            ok = self.inner.advance(min(now + 0.05, to_sim_time_s), max_steps)
            self._sample()
        return self.inner.observe_raw()["sim_time_s"] >= to_sim_time_s - 1e-12


if USE_FAKE:
    inner = FakeJointAdapter()
else:
    from ragleap_physical.adapters.mujoco import MujocoServoAdapter
    inner = MujocoServoAdapter()
rec = RecordingAdapter(inner)
bridge = Bridge(rec, Envelope(ENVELOPE), org_id="org_demo", max_capability=Capability.CONTROL)
server = make_server(PhysicalTools(bridge, proposer="employee:operations"))
URL = f"http://127.0.0.1:{server.server_address[1]}/mcp"
threading.Thread(target=server.serve_forever, daemon=True).start()
_id = [0]


def rpc(method, params=None):
    """Send one JSON-RPC message to the MCP server, like any MCP client would."""
    _id[0] += 1
    body = {"jsonrpc": "2.0", "id": _id[0], "method": method}
    if params is not None:
        body["params"] = params
    req = urllib.request.Request(URL, json.dumps(body).encode(), {"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return json.loads(r.read())["result"]


def tool(name, arguments=None):
    """Call one tool. Returns (payload, is_error)."""
    res = rpc("tools/call", {"name": name, "arguments": arguments or {}})
    return json.loads(res["content"][0]["text"]), res["isError"]


def show(title, obj=None):
    print(f"\n--- {title}")
    if obj is not None:
        print(json.dumps(obj, indent=2, sort_keys=True) if not isinstance(obj, str) else obj)


print("engine:", "built-in fake simulator" if USE_FAKE else "MuJoCo " + inner.observe_raw()["source"]["engine_version"])
print("MCP server is listening (local only):", URL)


## The agent's side (MCP)

In [ ]:
# AGENT SIDE: what an AI employee can do over MCP
init = rpc("initialize", {"protocolVersion": "2025-06-18", "capabilities": {}, "clientInfo": {"name": "demo", "version": "0"}})
show("STEP 1  initialize", {"server": init["serverInfo"], "protocol": init["protocolVersion"]})
show("STEP 2  tools the agent can see", [t["name"] for t in rpc("tools/list")["tools"]])

obs, err = tool("physical_get_observation")
e = obs["data"]["entities"][0]
show("STEP 3  observe (data is marked untrusted)", {"trust": obs["trust"], "simulated": obs["data"]["simulated"], "joint_1": {"pos": e["pos"], "vel": e["vel"]}})

env, _ = tool("physical_get_envelope")
show("STEP 4  the limits the agent can read but not change", env["data"]["targets"])

good = {"target": "joint_1", "command": "set_position_target", "value": 0.5, "unit": "rad",
        "reason": "Move joint_1 to 0.5 rad for the inspection step.",
        "expected_outcome": {"entity": "joint_1", "quantity": "pos", "lo": 0.45, "hi": 0.55, "unit": "rad", "by_sim_time_s": 5.0}}
prop, err = tool("physical_propose_action", good)
PID = prop["data"]["proposal"]["proposal_id"]
show("STEP 5  agent asks for a proposal", {"status": prop["data"]["status"], "proposal_id": PID, "digest": prop["data"]["digest"],
                                          "next": prop["data"]["next"], "joint_1 still at": tool("physical_get_observation")[0]["data"]["entities"][0]["pos"]})

attempts = {
    "value outside the limits (5.0 rad)": ("physical_propose_action", dict(good, value=5.0, expected_outcome=dict(good["expected_outcome"], lo=4.0, hi=6.0))),
    "a tool that applies the command": ("physical_apply", {"proposal_id": PID}),
    "choosing who is asking": ("physical_propose_action", dict(good, requested_by="person:alice")),
    "NaN value": ("physical_propose_action", dict(good, value=float("nan"))),
}
out = {}
for label, (name, args) in attempts.items():
    try:
        payload, is_err = tool(name, args)
        out[label] = ("refused: " + payload["error"]["code"]) if is_err else "ACCEPTED (unexpected)"
    except Exception as exc:  # a client library may also refuse NaN before sending
        out[label] = f"refused before sending ({type(exc).__name__})"
show("STEP 6  things the agent must NOT be able to do", out)


## The person's side (not reachable by the agent)

In [ ]:
# OPERATOR SIDE: a named person decides. This code is NOT an MCP tool and the agent cannot reach it.
proposal = bridge.get_proposal(PID)
before = rec.inner.observe_raw()["entities"][0]["pos"]
try:
    bridge.apply(proposal)
    r7 = "APPLIED WITHOUT APPROVAL (unexpected)"
except ApprovalError as exc:
    r7 = f"refused ({exc}); joint_1 still at {rec.inner.observe_raw()['entities'][0]['pos']}"
show("STEP 7  apply before anyone approves", r7)

print("\n--- STEP 8  person:alice reviews the exact proposal and approves it")
print("   proposal :", json.dumps({k: proposal.to_dict()[k] for k in ("target", "command", "params", "expected_outcome")}))
print("   digest   :", proposal.digest())
bridge.approve(proposal, "person:alice")
result = bridge.apply(proposal)
final = result.final_observation.entities[0]
show("STEP 8  result (the outcome was checked on the simulator state, not taken from the model)",
     {"state": result.state, "outcome_ok": result.outcome_ok, "joint_1 pos": round(final.pos, 4), "sim_time_s": result.final_observation.sim_time_s})

try:
    bridge.apply(proposal)
    r9 = "APPLIED TWICE (unexpected)"
except ApprovalError as exc:
    r9 = f"refused ({exc})"
show("STEP 9a replay of the same approval", r9)

second = dict(good, value=0.3, expected_outcome=dict(good["expected_outcome"], lo=0.25, hi=0.35, by_sim_time_s=10.0))
p2, _ = tool("physical_propose_action", second)
prop2 = bridge.get_proposal(p2["data"]["proposal"]["proposal_id"])
bridge.approve(prop2, "person:alice")
try:
    bridge.apply(dataclasses.replace(prop2, value=0.9))
    r9b = "APPLIED A CHANGED PROPOSAL (unexpected)"
except ApprovalError as exc:
    r9b = f"refused ({exc}); joint_1 still at {round(rec.inner.observe_raw()['entities'][0]['pos'], 4)}"
show("STEP 9b value changed from 0.3 to 0.9 after approval", r9b)

result2 = bridge.apply(prop2)
show("STEP 9c the original approved proposal (0.3) still works", {"state": result2.state, "joint_1 pos": round(result2.final_observation.entities[0].pos, 4)})

bridge.stop()
p3, _ = tool("physical_propose_action", dict(good, value=0.1, expected_outcome=dict(good["expected_outcome"], lo=0.05, hi=0.15, by_sim_time_s=15.0)))
prop3 = bridge.get_proposal(p3["data"]["proposal"]["proposal_id"])
bridge.approve(prop3, "person:alice")
try:
    bridge.apply(prop3)
    r10 = "APPLIED WHILE STOPPED (unexpected)"
except SafetyStop as exc:
    r10 = f"refused ({exc})"
show("STEP 10 stop (needs no approval) blocks even an approved proposal", r10)
bridge.clear_stop()


## Evidence

In [ ]:
# EVIDENCE: audit log and a plot of the real run
show("STEP 11 audit log (every step is recorded with hashes)", [f"{a['event']:<12} {a.get('proposal_id', ''):<12} {a.get('approver', a.get('code', ''))}" for a in bridge.audit])
try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    ts, ps = zip(*rec.trace)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(ts, ps, lw=2, label="joint_1 position (simulated)")
    for lo, hi, t0, t1, tgt in ((0.45, 0.55, 0, 5, 0.5), (0.25, 0.35, 5, 10, 0.3)):
        ax.fill_between([t0, t1], [lo, lo], [hi, hi], alpha=0.15, color="green")
        ax.hlines(tgt, t0, t1, colors="k", linestyles="--", lw=1)
    ax.set_xlabel("simulated time (s)"); ax.set_ylabel("position (rad)")
    ax.set_title("Approved commands on the simulator (dashed = target, green = expected band)", fontsize=10)
    ax.legend(loc="lower right"); fig.tight_layout(); fig.savefig("phase3-demo.png", dpi=120)
    print("\nplot saved as phase3-demo.png")
    try:
        from IPython.display import Image, display
        display(Image("phase3-demo.png"))
    except ImportError:
        pass  # not in a notebook: the file is still saved
except ImportError:
    print("(matplotlib is not installed: plot skipped)")
print("\nSimulation only. This shows the safety path works in a simulator. It says nothing about real machines.")
server.shutdown(); server.server_close()
